# Label-flipping poisoning, step by step

The **attacker** in our project: change some training labels from Attack (1) to Benign (0).
Run the cells one by one from the top. Each cell shows its result under it.

The real function lives in `src/poisoning.py`; the last part of this notebook checks it gives the same result.

## Cell 1: load the training labels

In [ ]:
import sys
from pathlib import Path
import numpy as np

# find the project folder (works whether this notebook is in the project root or a subfolder)
ROOT = Path.cwd()
while not (ROOT / "src").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
DATA = ROOT / "data" / "processed"
sys.path.insert(0, str(ROOT / "src"))          # so "import poisoning" works later

y_train = np.load(DATA / "y_train.npy")          # 0 = Benign, 1 = Attack
cat_train = np.load(DATA / "cat_train.npy", allow_pickle=True)   # category names

print("Data folder:", DATA)
print("Training rows:", len(y_train))
print("Benign rows:", (y_train == 0).sum(), "| Attack rows:", (y_train == 1).sum())

## Cell 2: the categories
How many training rows each category has.

In [ ]:
names, counts = np.unique(cat_train, return_counts=True)
for name, n in zip(names, counts):
    print(f"{name:<12} {n:>6}")

## Cell 3: find the pool (rows that MAY be flipped)
- **random**: any attack row
- **targeted**: only DDoS rows

Benign rows are never in the pool, so they can never be flipped.

In [ ]:
random_pool = np.flatnonzero(y_train == 1)                 # row numbers of every attack row
targeted_pool = np.flatnonzero(cat_train == "DDoS")          # row numbers of every DDoS row

print("random pool  :", len(random_pool), "rows")    # expected 72,800
print("targeted pool:", len(targeted_pool), "rows")  # expected 10,400
print("first 5 row numbers in the random pool:", random_pool[:5])

## Cell 4: how many labels to flip
The rate is a share of the **pool**, not of the whole training set.

In [ ]:
for rate in [0.05, 0.10, 0.20]:
    n_random = round(rate * len(random_pool))
    n_targeted = round(rate * len(targeted_pool))
    print(f"{rate:.0%}:  random {n_random:>6}  |  targeted {n_targeted:>5}")

## Cell 5: choose which rows (random 5% as the example)
Shuffle the pool with seed 42, then take the first n rows.
The same seed always gives the same rows, so results can be repeated.

In [ ]:
n = round(0.05 * len(random_pool))                   # 3,640
rng = np.random.default_rng(42)                         # seed 42
chosen = np.sort(rng.permutation(random_pool)[:n])      # shuffle, take the first n

print("rows chosen:", len(chosen))
print("first 5 chosen rows:", chosen[:5])
print("their labels before flipping:", y_train[chosen[:5]])   # all 1 (Attack)

## Cell 6: copy, then flip
We flip a **copy**, so the clean labels stay untouched for the clean baseline.

In [ ]:
y_poisoned = y_train.copy()
y_poisoned[chosen] = 0                                  # Attack (1) -> Benign (0)

print("labels after flipping:", y_poisoned[chosen[:5]])               # all 0 now
print("labels changed:", (y_train != y_poisoned).sum())               # 3,640
print("attack rows left:", (y_poisoned == 1).sum())                   # 72,800 - 3,640
print("original labels unchanged:", (y_train == 1).sum() == len(random_pool))

## Cell 7: now use the real function from `src/poisoning.py`
It does Cells 3 to 6 in one call. It should give exactly the same rows as above.

In [ ]:
from poisoning import poison_labels

y_p, flipped = poison_labels(y_train, cat_train, "random", 0.05, seed=42)
print("same rows as Cell 5:", np.array_equal(flipped, chosen))
print("same labels as Cell 6:", np.array_equal(y_p, y_poisoned))

## Cell 8: all 9 poisoning runs
- random and targeted at 5%, 10%, 20%
- equal-flip runs: random with exactly 520 / 1,040 / 2,080 flips, so both attacks change the same number of labels

In [ ]:
runs = [("random", 0.05, None), ("random", 0.10, None), ("random", 0.20, None),
        ("targeted", 0.05, None), ("targeted", 0.10, None), ("targeted", 0.20, None),
        ("random", None, 520), ("random", None, 1040), ("random", None, 2080)]

for strategy, rate, n_flips in runs:
    y_p, idx = poison_labels(y_train, cat_train, strategy, rate, seed=42, n_flips=n_flips)
    name = f"{strategy} {rate:.0%}" if n_flips is None else f"random, equal {n_flips}"
    print(f"{name:<20} flipped {len(idx):>6} | attacks left {(y_p == 1).sum():>6} | categories hit: {', '.join(sorted(set(cat_train[idx])))}")

## Cell 9: quick checks
The same things the tests check, shown as True/False.

In [ ]:
y_p, idx = poison_labels(y_train, cat_train, "targeted", 0.10, seed=42)
print("only attack rows were flipped:", (y_train[idx] == 1).all())
print("targeted flips are all DDoS:", (cat_train[idx] == "DDoS").all())
print("Benign rows never changed:", (y_p[y_train == 0] == 0).all())

_, a = poison_labels(y_train, cat_train, "random", 0.10, seed=42)
_, b = poison_labels(y_train, cat_train, "random", 0.10, seed=42)
print("same seed gives the same rows:", np.array_equal(a, b))

s05 = set(poison_labels(y_train, cat_train, "random", 0.05, seed=42)[1])
s10 = set(poison_labels(y_train, cat_train, "random", 0.10, seed=42)[1])
s20 = set(poison_labels(y_train, cat_train, "random", 0.20, seed=42)[1])
print("5% rows inside 10%, 10% inside 20% (nested):", s05 <= s10 <= s20)